In [12]:
import os
import sqlite3
import pandas as pd

# Automatically locate the directory containing the dataset files
dataset_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'olist_orders_dataset.csv' in files:
        dataset_dir = root
        break

if dataset_dir is None:
    raise FileNotFoundError("Could not find dataset files in /kaggle/input")

# Connect to SQLite database
conn = sqlite3.connect('olist.db')

# Read CSV files using the discovered path
orders = pd.read_csv(os.path.join(dataset_dir, 'olist_orders_dataset.csv'))
payments = pd.read_csv(os.path.join(dataset_dir, 'olist_order_payments_dataset.csv'))
items = pd.read_csv(os.path.join(dataset_dir, 'olist_order_items_dataset.csv'))

# Write tables to SQLite database
orders.to_sql('orders', conn, if_exists='replace', index=False)
payments.to_sql('payments', conn, if_exists='replace', index=False)
items.to_sql('items', conn, if_exists='replace', index=False)

print("Relational Database Connected! Tables 'orders', 'payments', and 'items' are live.")

Relational Database Connected! Tables 'orders', 'payments', and 'items' are live.


In [13]:
# Querying the database using intermediate SQL commands
query = """
SELECT 
    p.payment_type AS Payment_Method,
    COUNT(o.order_id) AS Total_Transactions,
    SUM(p.payment_value) AS Total_Revenue
FROM orders o
INNER JOIN payments p ON o.order_id = p.order_id
GROUP BY p.payment_type
ORDER BY Total_Revenue DESC;
"""

df_result = pd.read_sql_query(query, conn)
df_result

,Payment_Method,Total_Transactions,Total_Revenue
0,credit_card,76795,12542084.19
1,boleto,19784,2869361.27
2,voucher,5775,379436.87
3,debit_card,1529,217989.79
4,not_defined,3,0.00


In [14]:
# Querying top-performing items using an multi-table INNER JOIN
category_query = """
SELECT 
    i.price AS Product_Price,
    COUNT(o.order_id) AS Items_Sold
FROM orders o
INNER JOIN items i ON o.order_id = i.order_id
GROUP BY Product_Price
ORDER BY Items_Sold DESC
LIMIT 5;
"""

df_categories = pd.read_sql_query(category_query, conn)
df_categories

,Product_Price,Items_Sold
0,59.9,2481
1,69.9,1987
2,49.9,1953
3,89.9,1548
4,99.9,1432
